In [35]:
!pip install -U langchain-ollama langchain-openai


In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
!ls /content/drive/MyDrive/review_dataset


w_review_train.csv  w_review_train.parquet


In [38]:
import duckdb

con = duckdb.connect(database=':memory:', read_only=False)

In [39]:
df = con.execute("SELECT * FROM read_csv_auto('/content/drive/MyDrive/review_dataset/w_review_train.csv', parallel=false, encoding='UTF-8', ignore_errors=true)").fetchdf()

df.head()

,column0,column1
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,3
1,อาหารที่นี่เป็นอาหารจีนแคะที่หากินยากในบ้านเรา...,4
2,ปอเปี๊ยะสด ทุกวันนี้รู้สึกว่าหากินยาก (ร้านที่...,3
3,รัานคัพเค้กในเมืองไทยมีไม่มาก หลายๆคนอาจจะสงสั...,5
4,อร่อย!!! เดินผ่านDigital gatewayทุกวัน ไม่ยักร...,5


In [40]:
df.shape


(40000, 2)

In [41]:
con.execute("COPY df TO '/content/drive/MyDrive/review_dataset/w_review_train.parquet' (FORMAT PARQUET)")

In [42]:
parquet_avg_rating_direct = con.execute("SELECT AVG(column1) FROM '/content/drive/MyDrive/review_dataset/w_review_train.parquet' WHERE column0 LIKE '%อร่อย%'").fetchone()[0]
parquet_avg_rating_direct

3.8313027179006562

In [43]:
parquet_file_path = '/content/drive/MyDrive/review_dataset/w_review_train.parquet'


In [44]:
parquet_avg_rating_direct = con.execute(f"SELECT AVG(column1) FROM '{parquet_file_path}' WHERE column0 LIKE '%อร่อย%'").fetchone()[0]
parquet_avg_rating_direct

3.8313027179006562

In [45]:
coffee_keywords = [
    'กาแฟ', 'คาเฟ่', 'coffee', 'cafe', 'เอสเปรสโซ่', 'ลาเต้',
    'มอคค่า', 'คาปูชิโน่', 'เฟรนช์เพรส', 'cold brew', 'drip',
    'espresso', 'latte', 'americano', 'macchiato', 'brew',
    'specialty', 'slow bar', 'ร้านกาแฟ', 'ร้านคาเฟ่', 'กาแฟสด',
    'เมนูกาแฟ', 'กาแฟนม', 'กาแฟดำ', 'ร้านบาริสต้า', 'barista', 'coffee shop'
]

where_clause = " OR ".join([f"column0 LIKE '%{keyword}%'" for keyword in coffee_keywords])

In [46]:
%%time
coffee_reviews_df = con.execute(f"SELECT * FROM '{parquet_file_path}' WHERE {where_clause}").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 4.55 s, sys: 32.2 ms, total: 4.59 s
Wall time: 5.06 s


In [47]:
csv_file_path = '/content/drive/MyDrive/review_dataset/w_review_train.csv'


In [48]:
%%time
coffee_reviews_df_csv = con.execute(f"SELECT * FROM read_csv_auto('{csv_file_path}', parallel=false, encoding='UTF-8', ignore_errors=true) WHERE {where_clause}").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 2.66 s, sys: 50.1 ms, total: 2.71 s
Wall time: 5.88 s


In [49]:
coffee_reviews_df.head()


,column0,column1
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,3
1,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...,4
2,สารภาพว่าไม่เคยคิดจะไปต่อคิวซื้อมากินเองครับ บ...,3
3,ร้านอาหารญี่ปุ่นร้านนี้ ใจจริงไม่อยากแนะนำเลยค...,5
4,เดือนแรกที่เค้าต่อคิวกัน 2 - 3 ชั่วโมง เราก็ว่...,5


In [50]:
coffee_reviews_df.shape


(4981, 2)

In [51]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [52]:
!nohup ollama serve &


nohup: appending output to 'nohup.out'


In [53]:
!ollama pull gemma3:270m


In [54]:
!ollama list


NAME           ID              SIZE      MODIFIED               
gemma3:270m    e7d36fb2c3b3    291 MB    Less than a second ago    


In [55]:
!ollama run gemma3:270m "สวัสดี"


สวัสดีค่ะ ยินดีที่ได้รู้จักนะคะ 😊




In [56]:
!curl -s http://127.0.0.1:11434/api/tags


{"models":[{"name":"gemma3:270m","model":"gemma3:270m","modified_at":"2025-11-04T06:43:52.561363164Z","size":291554930,"digest":"e7d36fb2c3b3293cfe56d55889867a064b3a2b22e98335f2e6e8a387e081d6be","details":{"parent_model":"","format":"gguf","family":"gemma3","families":["gemma3"],"parameter_size":"268.10M","quantization_level":"Q8_0"}}]}

In [57]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [58]:
llm = ChatOllama(model="gemma3:270m")

In [59]:
response = llm.invoke("สวัสดี")
print(response)

content='สวัสดีค่ะ ยินดีที่ได้รู้จักนะคะ 😊\n' additional_kwargs={} response_metadata={'model': 'gemma3:270m', 'created_at': '2025-11-04T06:43:54.23003325Z', 'done': True, 'done_reason': 'stop', 'total_duration': 331313230, 'load_duration': 202986951, 'prompt_eval_count': 10, 'prompt_eval_duration': 8959151, 'eval_count': 12, 'eval_duration': 90811484, 'model_name': 'gemma3:270m', 'model_provider': 'ollama'} id='lc_run--79a79a66-be04-42f2-a589-56d5980c79a1-0' usage_metadata={'input_tokens': 10, 'output_tokens': 12, 'total_tokens': 22}


In [60]:
template = """
คุณเป็นครูสอน Python
คำถาม : {question}
"""

In [61]:
prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)

In [62]:
formatted_prompt = prompt.format(question="เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10")
print(formatted_prompt)


คุณเป็นครูสอน Python
คำถาม : เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10



In [63]:
chain = prompt | llm

In [64]:
response = chain.invoke({"question": "เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10"})
response

AIMessage(content='```python\ndef find_sum(numbers):\n    """\n    หาผลรวมของตัวเลข 1 ถึง 10\n    """\n    total = 0\n    for number in numbers:\n        total += number\n    return total\n\n# ตัวอย่างการใช้งาน\nnumbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]\nresult = find_sum(numbers)\nprint(result)\n```\n\n**คำอธิบายโค้ด:**\n\n1. **`def find_sum(numbers):`**:  กำหนดชื่อ funzione `find_sum` ที่จะเรียกใช้ ซึ่งจะถูกเรียกโดยใช้เมื่อมีการเรียกใช้งานโปรแกรม\n2. **`total = 0`**:  สร้างค่า `total` ที่จะเก็บผลรวมของตัวเลข\n3. **`for number in numbers:`**:  ใช้ `for` loop เพื่อวนซ้ำผ่านแต่ละตัวเลขใน `numbers`\n4. **`total += number`**:  เพิ่มค่า `number` เข้าไปใน `total`\n5. **`return total`**:  คืนค่า `total`\n6. **`numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`**:  สร้าง list ของตัวเลขที่ต้องการหาผลรวม\n7. **`result = find_sum(numbers)`**:  เรียกใช้ function `find_sum` เพื่อหาผลรวม\n8. **`print(result)`**:  แสดงผลลัพธ์ที่ได้จาก fungsi\n**ตัวอย่างการใช้งาน:**\n\n```python\nnumbers = [1, 2, 3, 4, 5, 6,

In [65]:
from IPython.display import display, Markdown


In [67]:
display(Markdown(response.content))

```python
def find_sum(numbers):
    """
    หาผลรวมของตัวเลข 1 ถึง 10
    """
    total = 0
    for number in numbers:
        total += number
    return total

# ตัวอย่างการใช้งาน
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
result = find_sum(numbers)
print(result)
```

**คำอธิบายโค้ด:**

1. **`def find_sum(numbers):`**:  กำหนดชื่อ funzione `find_sum` ที่จะเรียกใช้ ซึ่งจะถูกเรียกโดยใช้เมื่อมีการเรียกใช้งานโปรแกรม
2. **`total = 0`**:  สร้างค่า `total` ที่จะเก็บผลรวมของตัวเลข
3. **`for number in numbers:`**:  ใช้ `for` loop เพื่อวนซ้ำผ่านแต่ละตัวเลขใน `numbers`
4. **`total += number`**:  เพิ่มค่า `number` เข้าไปใน `total`
5. **`return total`**:  คืนค่า `total`
6. **`numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`**:  สร้าง list ของตัวเลขที่ต้องการหาผลรวม
7. **`result = find_sum(numbers)`**:  เรียกใช้ function `find_sum` เพื่อหาผลรวม
8. **`print(result)`**:  แสดงผลลัพธ์ที่ได้จาก fungsi
**ตัวอย่างการใช้งาน:**

```python
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
result = find_sum(numbers)
print(result)
```

**Output:**

```
15
```

**ข้อดีของโค้ดนี้:**

* **ใช้งานง่าย:**  โค้ดนี้เป็นโค้ดที่เข้าใจง่ายและใช้งานได้ง่าย
* **อ่านง่าย:**  โค้ดนี้มีโครงสร้างที่ชัดเจน
* **สามารถปรับปรุงได้:**  โค้ดนี้สามารถปรับปรุงได้โดยเพิ่มฟังก์ชันเพิ่มเติมเพื่อจัดการกับตัวอย่างที่กำหนดได้
* **สามารถอัปเดต:**  สามารถอัปเดตโค้ดได้โดยการเพิ่มฟังก์ชันเพิ่มเติมที่เกี่ยวข้องกับตัวเลขที่ต้องการ

**ข้อควรระวัง:**

* **ความปลอดภัย:**  ในบางกรณี คุณอาจพิจารณاةความปลอดภัยของโปรแกรมที่เขียนด้วย Python หากมีการใช้ข้อมูลที่อาจถูกรบกวน
* **การจัดการตัวแปร:**  ควรใช้ตัวแปรเพื่อจัดการตัวเลขที่ต้องการหาผลรวม
* **การใช้ `try...except`:**  หากตัวเลขที่ต้องการหาผลรวมไม่สามารถเขียนได้ในตัวแปร โปรดใช้ `try...except` เพื่อจัดการกับกรณีที่ตัวเลขไม่สามารถเขียนได้



In [68]:
prompt1 = PromptTemplate(
    template="แปลโจทย์เกี่ยวกับ {question}"
)

chain1 = prompt1 | llm
prompt2 = PromptTemplate(
    template="ทำความเข้าใจโจทย์ที่แปลแล้วต่อไปนี้ {translate}"
)

chain2 = prompt2 | llm
prompt3 = PromptTemplate(
    template="แก้โจทย์ตามที่เข้าใจต่อไปนี้ {understand}"
)

chain3 = prompt3 | llm
prompt4 = PromptTemplate(
    template="อธิบายวิธิีทำจากผลลัทธ์ต่อไปนี้ {solve}"
)

chain4 = prompt4 | llm

In [69]:
parser_chain = StrOutputParser()

full_chain = chain1 | chain2 | chain3 | chain4 | parser_chain

result = full_chain.invoke({"question":"2x + 5 = 15"})

result

'ถูกต้องแล้วครับ! การแปลโจทย์ที่ "2x + 5 = 15" เป็นความหมายถูกต้องและเป็นภาษาที่เข้าใจง่ายที่สุด โดยการใช้ผลลัทธ์ที่เน้นไปที่การ **จัดเรียงลำดับความสำคัญของคำต่างๆ ในโจทย์** เพื่อให้เข้าใจความหมายของคำแต่ละคำได้อย่างชัดเจน'

In [70]:
import os
from langchain_openai import ChatOpenAI

In [71]:
OPENAI_API_KEY = "xxx"

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [72]:
llm2 = ChatOpenAI(model="gpt-4o-mini")

In [73]:
chain1 = prompt1 | llm2
chain2 = prompt2 | llm2
chain3 = prompt3 | llm2
chain4 = prompt4 | llm2

full_chain = chain1 | chain2 | chain3 | chain4 | parser_chain

In [75]:
formatted_result = result.replace('\\[', '$$') \
                         .replace('\\]', '$$') \
                         .replace('\\(', '$') \
                         .replace('\\)', '$')
display(Markdown(formatted_result))

ถูกต้องแล้วครับ! การแปลโจทย์ที่ "2x + 5 = 15" เป็นความหมายถูกต้องและเป็นภาษาที่เข้าใจง่ายที่สุด โดยการใช้ผลลัทธ์ที่เน้นไปที่การ **จัดเรียงลำดับความสำคัญของคำต่างๆ ในโจทย์** เพื่อให้เข้าใจความหมายของคำแต่ละคำได้อย่างชัดเจน

In [76]:
prompt_template = """
[1. Role/Context บอก LLM ว่าควรทำตัวเป็นอะไร]
คุณเป็น AI ที่เชี่ยวชาญในการวิเคราะห์รีวิวภาษาไทย

[2. Task Description บอกงานที่ต้องทำ]
จาก Review ต่อไปนี้ ให้ Extract 3 keywords
ที่เป็นตัวแทนของ Review นี้ได้ดีที่สุด

[3. Input Data ข้อมูลที่ต้องประมวลผล]
Review: {review}

[4. Output Format บอกรูปแบบของคำตอบ]
ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'keywords'
เช่น {{"keywords": "กาแฟ, บรรยากาศ, ราคา"}

[5. Constraints ข้อจำกัดหรือเงื่อนไข]
- ต้องเป็นคำภาษาไทย
- คั่นด้วย comma
- หากไม่พบให้ตอบ 'ไม่ระบุ'

[6. Examples (optional) ตัวอย่าง input/output]
ตัวอย่าง
Input: "กาแฟอร่อย บรรยากาศดี ราคาแพง"
Output: {"keywords": "กาแฟ, บรรยากาศ, ราคา"}
"""

In [77]:
prompt = """
Extract keywords from this review
{review}
"""

# Pros Simple, fast
# Cons อาจได้ผลไม่ดีกับ Tasks ซับซ้อน

In [78]:
prompt = """
Extract keywords from reviews.

Example 1
Review: "กาแฟอร่อย บริการดี"
Keywords: กาแฟ, บริการ

Example 2
Review: "ราคาแพง แต่คุ้มค่า"
Keywords: ราคา, คุณภาพ

Now extract from this review
Review: {review}
Keywords:
"""

# Pros ได้ผลดีกว่า Zero-shot
# Cons Prompt ใช้ Tokens มากขึ้น

In [79]:
prompt = """
Review: {review}

ขั้นตอนการ Extract keywords
1. อ่านรีวิวและเข้าใจความหมาย
2. ระบุหัวข้อหลักที่กล่าวถึง
3. เลือก 3 keywords ที่สำคัญที่สุด
4. ตอบในรูปแบบ JSON: {{"keywords": "คำ1, คำ2, คำ3"}}

ให้ทำทีละขั้นตอน
"""

# Pros ได้ผลดีกับ Reasoning tasks
# Cons ใช้ Tokens มากกว่า ช้ากว่า

In [80]:
keyword_prompt = PromptTemplate(
template = """
จาก Review ต่อไปนี้
Extract 3 keywords ที่เป็นตัวแทนของ Review นี้

Review: {review}

ขั้นตอนการ Extract keywords
1. อ่านรีวิวและเข้าใจความหมาย
2. ระบุหัวข้อหลักที่กล่าวถึง
3. เลือก 3 keywords ที่สำคัญที่สุด
4. ตอบในรูปแบบ JSON: {{"keywords": "คำ1, คำ2, คำ3"}}

ให้ทำทีละขั้นตอน
""")

In [81]:
sample_reviews = coffee_reviews_df['column0'].head(2)
sample_reviews

,column0
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...
1,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...


In [82]:
sample_reviews.to_csv('/content/sample_reviews.csv', index=False)

In [83]:
keyword_extract = keyword_prompt | llm | StrOutputParser()

In [84]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    cleaned = (text).replace("**", "").strip()
    results.append({"review": review, "text": cleaned})

CPU times: user 124 ms, sys: 7.79 ms, total: 132 ms
Wall time: 3.53 s


In [87]:
for result in results:
    print(f"Review {result['review'][:1000]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย 
ลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่
ทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล 
พะแนงห่อไข่อร่อยดี เห้ยแต่ราคา 150บาทมันเกินไปนะ รับไม่ไหวว
เลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย )ขนมไทย) 

คงไม่ไปซ้ำ แพงเกิน ...
text ```json
{"keywords": ["ร้านอาหาร", "ห้องน้ำ", "กาแฟ", "น้ำผึ้ง", "น้ำผึ้ง", "ลาบไข่ต้ม", "ไข่มัน", "ขนมไทย", "ขนมหวาน", "ฟรี", "ราคา", "แพง", "เกินไป", "รับไม่ไหว", "เลิกกิน", "ขนมหวาน", "ขนมไทย", "ร้านอาหาร", "อาหารใหญ่", "ราคาสูง", "ไม่ไปซ้ำ", "ราคาก็เกินไป", "รับไม่ไหว", "กินแล้วมีขนมหวานให้กินฟรี", "ขนมไทย", "ร้านอาหาร", "อาหาร", "อาหารใหญ่", "อาหารทะเล", "อาหารทะเลสดๆ", "ไข่ต้ม", "ไข่มัน", "น้ำผึ้ง", "น้ำผึ้งมาราด", "ไข่มันคาว", "ทำไมไม่ไปซ้ำ", "ราคาก็เกินไป", "ขั้นตอนการ Extract keywords": "1. อ่านรีวิวและเข้าใจความหมาย"
```

Review วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์

ก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่

In [86]:
keyword_extract = keyword_prompt | llm2 | StrOutputParser()

In [89]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

In [90]:
!ollama pull scb10x/typhoon2.1-gemma3-4b

In [91]:
!ollama list


NAME                                  ID              SIZE      MODIFIED               
scb10x/typhoon2.1-gemma3-4b:latest    8cfab9097c9d    2.6 GB    Less than a second ago    
gemma3:270m                           e7d36fb2c3b3    291 MB    4 minutes ago             


In [94]:
llm3 = ChatOllama(
    model="scb10x/typhoon2.1-gemma3-4b",
  )

In [95]:
keyword_extract = keyword_prompt | llm3 | StrOutputParser()


In [96]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    cleaned = (text).replace("**", "").strip()
    results.append({"review": review, "text": cleaned})

CPU times: user 231 ms, sys: 9.95 ms, total: 240 ms
Wall time: 18.4 s


In [97]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ...
text แน่นอนครับ ผมจะช่วยคุณวิเคราะห์และดึงคำสำคัญจากรีวิวนี้ออกมาเป็น JSON ตามที่คุณต้องการ

ขั้นตอนที่ 1: อ่านรีวิวและเข้าใจความหมาย

รีวิวนี้พูดถึงร้านอาหารขนาดใหญ่ที่มีข้อดีและข้อเสียหลายอย่าง ได้แก่:

*   ขนาดร้านใหญ่
*   ห้องน้ำก่อนเข้าทาง
*   อาหารบางอย่าง (ลาบไข่ต้ม, ทอดมันหัวปลี, พะแนงห่อไข่)
*   ราคาอาหารที่สูงเกินไป
*   มีของหวานให้ฟรี

ขั้นตอนที่ 2: ระบุหัวข้อหลักที่กล่าวถึง

*   ขนาดร้าน
*   คุณภาพอาหาร (โดยเฉพาะอาหารไทย)
*   ราคา

ขั้นตอนที่ 3: เลือก 3 keywords ที่สำคัญที่สุด

จากหัวข้อหลักที่กล่าวถึง ผมเลือกคำเหล่านี้เป็น keywords ที่สำคัญที่สุด:

1.  ร้านอาหาร (ขนาดใหญ่)
2.  ราคาแพง
3.  อาหารไทย

ขั้นตอนที่ 4: ตอบในรูปแบบ JSON

```json
{
  "keywords": ["ร้านอาหาร", "ราคาแพง", "อาหารไทย"]
}
```

หวังว่าคำตอบนี้จะเป็นประโยชน์นะครับ หากคุณมีคำถามเพิ่มเติมหรือต้องการให้ผมช่วยวิเคราะห์อะไรอีก โปรดแจ้งให้ทราบได้เลย

Review วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอ

In [98]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

class ReviewKeywords(BaseModel):
    keywords: str = Field(
        ...,  # Required field
        description="3 keywords separated by comma",
        examples=["กาแฟ, บรรยากาศ, ราคา"]
    )

class KeywordsExtraction(BaseModel):
    extracted_qualities: ReviewKeywords

In [99]:
keywords_parser = JsonOutputParser(pydantic_object=KeywordsExtraction)


In [100]:
keyword_extract = keyword_prompt | llm3 | keywords_parser


In [101]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    results.append({"review": review, "text": text})

CPU times: user 264 ms, sys: 15.1 ms, total: 279 ms
Wall time: 15.4 s


In [102]:
results[0]['text']['keywords']

['ราคา', 'อาหารไทย', 'รสชาติ']

In [103]:
results[1]['text']['keywords']


['กาแฟ', 'บรรยากาศ', 'บริการ']

In [105]:
llm3 = ChatOllama(
    model="scb10x/typhoon2.1-gemma3-4b",
    temperature=0.05
)

keyword_extract = keyword_prompt | llm3 | keywords_parser

In [106]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    results.append({"review": review, "text": text})

CPU times: user 308 ms, sys: 23.9 ms, total: 332 ms
Wall time: 14.5 s


In [107]:
results[0]['text']['keywords'], results[1]['text']['keywords']

(['ราคา', 'รสชาติ', 'ขนาด'], ['กาแฟ', 'บรรยากาศ', 'รสชาติ'])

In [108]:
keyword_prompt = PromptTemplate(
    template="""จาก Review ต่อไปนี้
    Extract keywords 3 keywords เท่านั้น ที่เป็นตัวแทนของ Review นี้ได้ดีที่สุด

    Review: {review}

    ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'keywords'
    และค่าเป็นข้อความที่ดึงได้คั่นด้วย comma
    เช่น {{"keywords": "กาแฟ, บรรยากาศ, ราคา"}}

    หากไม่พบข้อมูลให้ตอบ 'ไม่ระบุ'
    """
)

In [109]:
keyword_extract = keyword_prompt | llm3 | keywords_parser


In [110]:
inputs = [{"review": r} for r in sample_reviews]

In [111]:
%%time
results = keyword_extract.batch(inputs)
results


CPU times: user 30.6 ms, sys: 4.29 ms, total: 34.9 ms
Wall time: 2.39 s


[{'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
 {'keywords': 'กาแฟ, บรรยากาศ, รสชาติ'}]

In [112]:
sample_reviews_1000 = coffee_reviews_df['column0'].head(1000).tolist()
inputs_1000 = [{"review": r} for r in sample_reviews_1000]

BATCH_SIZE = 8

results_1000 = []

In [113]:
from tqdm import tqdm


In [114]:
for i in tqdm(range(0, len(inputs_1000), BATCH_SIZE), desc="Extracting keywords"):
    batch = inputs_1000[i:i+BATCH_SIZE]
    outputs = keyword_extract.batch(batch)
    for inp, out in zip(batch, outputs):
        results_1000.append({"review": inp["review"], "text": out})

Extracting keywords: 100%|██████████| 125/125 [18:42<00:00,  8.98s/it]


In [115]:
for result in results_1000[0:1000]:
    print(result['text']['keywords'])

ราคา, บรรยากาศ, ไข่ต้ม
กาแฟ, บรรยากาศ, รสชาติ
โดนัท, น้ำตาล, รสชาติ
ปลาซาบะ, ไข่ตุ๋น, ราคาคุ้มค่า
โดนัท, Original Glazed, คิว
วังพญาไท, นรสิงห์, คาเฟ่
อาหารทะเล, ราคา, บรรยากาศ
กาแฟ, เค้ก, พนักงาน
Shibuya Honey Toast, ฮันนี่ โทสท์, ราคา
คาเฟ่, วานิลลา, คาโบนาร่า
กาแฟ, เค้ก, สมูตตี้
กาแฟ, บรรยากาศ, สลัดไก่
ไอศครีม, ชิม, ราคา
ก๋วยเตี๋ยว, เอกมัย, ปอเปี๊ยะ
ข้าวห่อไข่, เมนู, บรรยากาศ
หวาน, ช็อกโกแลต, เครื่องใช้
บรรยากาศ, ปลั๊กไฟ, กาแฟ
กาแฟ, อาหาร, โปรโมชั่น
ปลาเผา, น้ำจิ้ม, แกงเลียง
ติ่มซำ, 24 ชั่วโมง, ราคา
รสชาติ, บรรยากาศ, สเต็ก
ราคา, พาสต้า, บรรยากาศ
พิซซ่า, คาลามารี, ซัลมอน
บรรยากาศ, ราคา, ข้าวผัด
กาแฟ, บรรยากาศ, ราคา
บรรยากาศ, อาหาร, ที่จอดรถ
ขนมสายไหม, รสชาติ, บรรยากาศ
กาแฟ, ราคา, บริการ
บริการ, พนักงาน, จานเปล่า
บรรยากาศ, ขนมปัง, ชา
ชาบู, ราคา, บรรยากาศ
โรงพยาบาล, บรรยากาศ, อาหาร
ข้าวห่อไข่, เมนูหลากหลาย, ราคา
ติ่มซํา, ขนมผักกาด, ปลาเต้าซี่
แกะ, เนื้อแกะ, รสชาติ
เค้ก homemade, mocha choc cake, เค้กแครอท
ปากน้ำ, กาแฟ, ร้าน Chubby
ร้านกาแฟ, บรรยากาศเงียบสงบ, ราคาไม่แพง
ราคา, ความหลากหล

In [116]:
context_prompt = PromptTemplate(
    template="""จาก Review ต่อไปนี้
    Extract context ใน Review ที่เป็นตัวแทนที่บรรยายคำว่า {keyword} ได้ดีที่สุด

    Review: {review}

    ตอบผลลัพธ์ในรูปแบบ JSON โดยมีคีย์เป็น 'context'
    และค่าเป็นข้อความที่ดึงได้จาก Review

    ตัวอย่าง
    keyword คือ "กาแฟ"
    context: "กาแฟที่นี่อร่อยมาก หอมกลิ่นกาแฟคั่วสด"

    หากไม่พบข้อมูลให้ตอบ 'ไม่ระบุ'
    """
)

In [117]:
sentiment_prompt = PromptTemplate(
    template="""จากข้อความต่อไปนี้ ให้วิเคราะห์ว่าเป็น sentiment แบบไหน
    เลือกจากตัวเลือกเหล่านี้เท่านั้น: "positive", "negative", "neutral"

    ข้อความ: {message}

    ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'sentiment'
    และค่าเป็นหนึ่งใน list ต่อไปนี้ [positive, negative, neutral]

    คำแนะนำ
    - positive คือ ข้อความที่แสดงความพอใจ ชอบ ดี
    - negative คือ ข้อความที่แสดงความไม่พอใจ ไม่ชอบ แย่
    - neutral คือ ข้อความที่ไม่แสดงความรู้สึกชัดเจน เป็นกลาง

    # ตัวอย่าง
    # ข้อความ "อร่อยมาก" → {{"sentiment": "positive"}}
    # ข้อความ "แพงไป" → {{"sentiment": "negative"}}
    # ข้อความ "ปกติ" → {{"sentiment": "neutral"}}
    """
)

In [118]:
class ContextKeyword(BaseModel):
    context: str = Field(..., description="context that best represent this keyword")

class ContexExtraction(BaseModel):
    extracted: ContextKeyword

context_parser = JsonOutputParser(pydantic_object=ContexExtraction)

In [119]:
context_extract = context_prompt | llm3 | context_parser

In [120]:
class Sentiment(BaseModel):
    sentiment: str = Field(..., description="sentiment of review")

class SentimentExtraction(BaseModel):
    extracted: Sentiment


sentiment_parser = JsonOutputParser(pydantic_object=SentimentExtraction)

In [121]:
sentiment_extract = sentiment_prompt | llm3 | sentiment_parser

In [122]:
results_1000[0]['text']['keywords'].split(',')


['ราคา', ' บรรยากาศ', ' ไข่ต้ม']

In [123]:
context_inputs = []

for review_data in tqdm(results_1000, desc="Preparing context inputs"):
    review = review_data['review']
    keywords = review_data['text']['keywords'].split(',')
    for keyword in keywords:
        context_inputs.append({"review": review, "keyword": keyword.strip()})

Preparing context inputs: 100%|██████████| 1000/1000 [00:00<00:00, 358702.13it/s]


In [124]:
len(context_inputs)


3000

In [125]:
context_results = []
for i in tqdm(range(len(results_1000)), desc="Extracting contexts"):
   result = context_extract.batch(context_inputs[i*3:(i*3)+3])
   contexts_list = [d['context'] for d in result]
   results_1000[i]['contexts'] = contexts_list

Extracting contexts: 100%|██████████| 1000/1000 [1:13:07<00:00,  4.39s/it]


In [126]:
results_1000[:5]


[{'review': 'ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี \nห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย \nลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่\nทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล \nพะแนงห่อไข่อร่อยดี เห้ยแต่ราคา 150บาทมันเกินไปนะ รับไม่ไหวว\nเลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย )ขนมไทย) \n\nคงไม่ไปซ้ำ แพงเกิน ',
  'text': {'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
  'contexts': ['ราคา 150 บาทมันเกินไปนะ รับไม่ไหวว',
   'บรรยากาศร้านใหญ่มากกกกกกก',
   'ไข่ต้ม']},
 {'review': 'วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์\n\nก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก Twitter ว่ากาแฟที่นี่อร่อยมากกกกกกก\nเรียกว่าระดับแฟนๆ Starbuck ยังต้องเหลียวมามอง ก็ดองมานานจนถึงวันนี้โอกาสเหมาะ ไป\npresent งานที่ตึก Software Park เสร็จก็เลยมาหาไรกินแถวนี้พอดี\nจัดไป.....\n\n\nบรรยากาศรอบๆร้านก็แต่งได้น่านั่ง อารมณ์ประมาณว่าอยู่ในสวนบ้านเราเอง ต้นไม้เยอะดี\nอย่างน้อยก็เป็นสีเขียวเล็กๆ ในใจกลางเมืองได้ละ\nเขียวชอุ่ม\n\nว่าแล้วไม่รอช้

In [127]:
sentiment_inputs = []

for review_data in tqdm(results_1000, desc="Preparing sentiment inputs"):
    contexts = review_data['contexts']
    for context in contexts:
        sentiment_inputs.append({"message": context})

Preparing sentiment inputs: 100%|██████████| 1000/1000 [00:00<00:00, 288426.90it/s]


In [128]:
len(sentiment_inputs)


3000

In [129]:
sentiment_inputs[:6]


[{'message': 'ราคา 150 บาทมันเกินไปนะ รับไม่ไหวว'},
 {'message': 'บรรยากาศร้านใหญ่มากกกกกกก'},
 {'message': 'ไข่ต้ม'},
 {'message': 'กาแฟที่นี่อร่อยมากกกกกกก เรียกว่าระดับแฟนๆ Starbuck ยังต้องเหลียวมามอง'},
 {'message': 'บรรยากาศรอบๆร้านก็แต่งได้น่านั่ง อารมณ์ประมาณว่าอยู่ในสวนบ้านเราเอง ต้นไม้เยอะดี อย่างน้อยก็เป็นสีเขียวเล็กๆ ในใจกลางเมืองได้ละ เขียวชอุ่ม'},
 {'message': 'รสชาติและกลิ่นมันไม่ธรรมดาเลย หอมๆ ถือว่าเป็นมวยรุ่นเดียวกะ Starbuck ได้'}]

In [130]:
for i in tqdm(range(len(results_1000)), desc="Extracting sentiment"):
   result = sentiment_extract.batch(sentiment_inputs[i*3:(i*3)+3])
   sentiment_list = [d['sentiment'] for d in result]
   results_1000[i]['sentiment'] = sentiment_list

Extracting sentiment: 100%|██████████| 1000/1000 [44:11<00:00,  2.65s/it]


In [131]:
results_1000[0:5]


[{'review': 'ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี \nห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย \nลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่\nทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล \nพะแนงห่อไข่อร่อยดี เห้ยแต่ราคา 150บาทมันเกินไปนะ รับไม่ไหวว\nเลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย )ขนมไทย) \n\nคงไม่ไปซ้ำ แพงเกิน ',
  'text': {'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
  'contexts': ['ราคา 150 บาทมันเกินไปนะ รับไม่ไหวว',
   'บรรยากาศร้านใหญ่มากกกกกกก',
   'ไข่ต้ม'],
  'sentiment': ['negative', 'positive', 'neutral']},
 {'review': 'วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์\n\nก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก Twitter ว่ากาแฟที่นี่อร่อยมากกกกกกก\nเรียกว่าระดับแฟนๆ Starbuck ยังต้องเหลียวมามอง ก็ดองมานานจนถึงวันนี้โอกาสเหมาะ ไป\npresent งานที่ตึก Software Park เสร็จก็เลยมาหาไรกินแถวนี้พอดี\nจัดไป.....\n\n\nบรรยากาศรอบๆร้านก็แต่งได้น่านั่ง อารมณ์ประมาณว่าอยู่ในสวนบ้านเราเอง ต้นไม้เยอะดี\nอย่างน้อยก็เป็นสีเขียวเล

In [132]:
import pandas as pd

df = pd.DataFrame([
    {
        'review': r['review'],
        **{f'keyword{i+1}': k.strip() for i, k in enumerate(r['text']['keywords'].split(','))},
        **{f'context{i+1}_keyword': c for i, c in enumerate(r['contexts'])},
        **{f'sentiment{i+1}': s for i, s in enumerate(r['sentiment'])}
    }
    for r in results_1000
])

In [133]:
df.to_csv('results.csv', index=False, encoding='utf-8-sig')
